# ESOREX quickstart: predicting TyrB substrate specificity

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UCB-BioE-Anderson-Lab/ESOREX/blob/main/notebooks/tyrb_quickstart.ipynb)

This notebook trains **ESOREX** on the 9 natural amino-acid substrates of *E. coli* **TyrB**
(an aromatic-preferring aminotransferase) and predicts its activity on 14 held-out *unnatural*
analogs. For every prediction it reports whether the value is **determined** by the training
data or an **extrapolation** beyond it.

ESOREX converts measured rates to activation free energies (`E = -RT ln k`) and solves a
free-energy decomposition as a hard constraint that reproduces the training rates exactly.

Runs in about a minute on a free Colab CPU. No GPU needed.

## 1. Install ESOREX and fetch the data

The energetic model needs only RDKit, NumPy and SciPy. We clone the repository to get both
the `esorex` package and the curated TyrB dataset.

In [ ]:
!pip install -q rdkit          # numpy, scipy and pandas are preinstalled in Colab (do not downgrade them)
!git clone -q https://github.com/UCB-BioE-Anderson-Lab/ESOREX.git
import sys
sys.path.insert(0, "/content/ESOREX")
DATA = "/content/ESOREX/data/transaminases/ONUFFER_curated.csv"

## 2. Load the measured substrates

Onuffer & Kirsch measured TyrB (the eTATase variant) against a panel of amino acids. Each
substrate is a SMILES with a measured rate. The **reactive core** (the amine, alpha-carbon,
and carboxyl the enzyme acts on) is matched with a SMARTS pattern; specificity is learned
from everything *outside* that core. We split the panel into the 9 naturals TyrB evolved with
(training) and everything else (held-out unnatural analogs).

In [ ]:
import csv
from rdkit import Chem

RATE_COL  = "eTATase_kf_over_KD_M-1_s-1"
DUPLICATE = {"Arginine (mu = 1.0)"}
NATURALS  = {"Aspartate", "Glutamate", "Phenylalanine", "Tryptophan", "Tyrosine",
             "Alanine", "Leucine", "Valine", "Arginine (mu = 0.2)"}
AA_CORE   = Chem.MolFromSmarts("[NX3][CX4H][CX3](=O)[OX2]")

def core(mol):
    """Reaction-center atoms (amine, alpha-C, carboxyl); specificity acts on the rest."""
    return set(mol.GetSubstructMatch(AA_CORE))

def parse_rate(s):
    s = s.strip()
    if not s or s.lower() in ("nd", "n/a", "na", "-"): return None
    try: return float(s)
    except ValueError: return None

naturals, analogs = [], []
with open(DATA) as f:
    for row in csv.DictReader(f):
        name = row["substrate"].strip()
        if name in DUPLICATE: continue
        rate = parse_rate(row[RATE_COL])
        mol  = Chem.MolFromSmiles(row.get("substrate_smiles", "").strip())
        if rate is None or mol is None or not core(mol): continue
        (naturals if name in NATURALS else analogs).append((name, mol, rate))

print(f"training naturals: {len(naturals)}    held-out analogs: {len(analogs)}")

## 3. Train the energetic model

Training solves `X w = E` as a hard constraint. `exact = True` means the model reproduces
every training rate exactly (adding cooperative terms only if the additive model cannot).

In [ ]:
from esorex.energetic_specificity import EnergeticSpecificityModel

model = EnergeticSpecificityModel()
info = model.train([m for _, m, _ in naturals],
                   [core(m) for _, m, _ in naturals],
                   rates=[r for _, _, r in naturals])
print("trained on", info["n_substrates"], "substrates | exact fit:", info["exact"])

## 4. Predict the held-out analogs, with provenance

`model.predict` returns a predicted rate plus a verdict: **determined** (pinned by the data,
identical across every exact-fitting solution) or **extrapolation** (with a novelty score for
how far outside the training support it reaches).

In [ ]:
import pandas as pd
from scipy.stats import spearmanr

rows = []
for name, mol, measured in analogs:
    p = model.predict(mol, core(mol))
    rows.append(dict(substrate=name,
                     measured_rate=measured,
                     predicted_rate=round(p.rate, 1),
                     provenance="determined" if p.determined else "extrapolation",
                     novelty=round(p.novelty, 3)))

df = pd.DataFrame(rows).sort_values("predicted_rate", ascending=False).reset_index(drop=True)
rho = spearmanr(df.measured_rate, df.predicted_rate).correlation
print(f"held-out Spearman rho = {rho:.2f}   (n = {len(df)})")
df

## 5. Reading the result

The model ranks the held-out analogs well (Spearman rho about 0.73). Note that most
predictions are marked **extrapolation**: with only 9 training substrates, the data pin down
few directions exactly, and ESOREX says so rather than feigning confidence.

The most novel analog, **2-aminooctanoate** (a long aliphatic side chain with no aromatic
ring, a combination absent from the naturals), is the model's clearest miss and is flagged
accordingly, not reported as a confident prediction.

In [ ]:
df[df.provenance == "extrapolation"].sort_values("novelty", ascending=False).reset_index(drop=True)

## Where to go next

- The same model handles **regioselectivity** (picking one site on a scaffold): see the
  FucTIII demonstration in `docs/demonstrations/`.
- Concept pages: `docs/theory.md`, `docs/representation.md`, `docs/model.md`.
- ESOREX trains **one model per enzyme / reaction type**; point it at your own atom-mapped
  reactions and measured rates to model a different enzyme.